# 17 — Covariate builders: self-checks and artifact verification

Four modules in `src/` are invoked from the command line rather than from a notebook:

| module | artifact | what it is |
|---|---|---|
| `src/burn_history.py` | `data/hex_burn_history.parquet` | prior-burn **state** per hex-season |
| `src/hex_ignitions.py` | `data/hex_ignitions.parquet` | ignition counts per hex-season, by cause class |
| `src/hex_climate.py` | `data/hex_season_climate.parquet` | TerraClimate covariates re-fetched at hex grain |
| `src/hex_ndvi.py` | `data/hex_season_ndvi.parquet` | MODIS NDVI/EVI fuel load, six forest ecoregions |

That was a deliberate choice — three of the four are long resumable network or
disk jobs that have no business living in a notebook cell — but it left the
project with code no notebook traces. Downstream notebooks (`12`–`15`) read the
parquets these modules write and never name the modules themselves, so the
reasoning that produced those columns was invisible from the notebook side.

**This notebook closes that gap without re-running the builds.** It does two
things per module:

1. **Runs the module's `_self_check()` for real.** These are the leakage and
   densification assertions that guard the DJF boundary rule and the pre-season
   window rule. They run on synthetic frames in memory, cost nothing, and are
   the part that actually needs to be seen.
2. **Verifies the shipped artifact** against the contract the module promises —
   shape, schema, spine, and the specific invariant each build is supposed to
   hold.

The `build()` calls themselves appear in each section as a guarded cell that
does not execute. Re-running them is a deliberate act: `hex_climate` and
`hex_ndvi` are multi-hour STAC/THREDDS fetches, and `burn_history` and
`hex_ignitions` overwrite artifacts that notebooks 12–15 and every W6 figure
depend on.

> **Read this as verification, not derivation.** Nothing here re-derives a
> finding. The claims about what these covariates *mean* — the five ignition
> nulls, the climate + NDVI acres gain — are settled in notebooks 12–14.

## Setup

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
import burn_history as bh
import hex_climate as hc
import hex_ignitions as hi
import hex_ndvi as hn
import terraclimate as tc
from config import ProjectConfig

warnings.filterwarnings("ignore")
cfg = ProjectConfig()
DATA = cfg.data

# Set True only if you intend to re-run the builds. See the guarded cells below
# for what that costs; the artifacts are inputs to notebooks 12-15.
REBUILD = False

print(f"data dir: {DATA}")
print(f"REBUILD:  {REBUILD}")

In [ ]:
def verify(name, df, *, expect_cols, key=("hex_id", "season_idx")):
    """Report shape and schema, and assert the artifact's structural contract.

    Every one of these builds emits a densified hex x season panel keyed the same
    way, so the key-uniqueness and null-key checks are shared. Anything specific
    to one artifact is asserted in its own section.
    """
    print(f"{name}: {len(df):,} rows x {df.shape[1]} cols")

    missing = [c for c in expect_cols if c not in df.columns]
    assert not missing, f"{name} is missing promised columns: {missing}"

    key = list(key)
    assert not df[key].isna().any().any(), f"{name} has null keys"
    dupes = df.duplicated(subset=key).sum()
    assert dupes == 0, f"{name} has {dupes:,} duplicate {key} rows"

    print(f"  key {tuple(key)} unique, no nulls")
    print(f"  cols: {list(df.columns)}")
    return df

---
## 1 — `src/burn_history.py` — prior-burn state

Prior burn is a **state, not a forecast**: whether a hex burned in an earlier
season is known with certainty before the target season opens. That makes it
legitimate as a feature in a way a weather forecast would not be — but only if
the lag is right, which is exactly what the self-check asserts.

Two module decisions worth surfacing here, because they are invisible in the
parquet:

- **Point-only fires are excluded on semantic, not quality, grounds.** A 14-acre
  point fire against a 62,494-acre hex is 0.02% of the cell. Including it would
  make the feature encode *where small fires get reported*, which near-leaks the
  ignition target this covariate is meant to predict.
- **Burned fraction is clamped at 1.0.** Genuine full burns and
  boundary-clipped partial hexes are indistinguishable afterward (0.759% of
  perimeter hex-years).

### 1.1 Self-check — the leakage assertion

`trailing_burn` must produce a lag that a season cannot see through. A hex that
burns at `season_idx = 2` must read `burned_frac_lag4 == 0.0` **at t=2 itself**
and `0.5` at t=3. Getting this wrong is the silent failure mode: the feature
would read the burn scar it is supposed to predict, and the result would look
excellent.

In [ ]:
bh._self_check()

### 1.2 Verify the shipped artifact

The panel is densified — every hex appears in every season, including hexes that
never burned — so the row count is `hexes x seasons` exactly, not a count of
burn events. The `any_burn_lag*` prevalences below are the ones quoted in
`CLAUDE.md`: low, because this is perimeter-backed burn only.

**One subtlety about the clamp, worth stating because it invites a wrong
assertion.** `burned_frac` is clamped to 1.0 — but that clamp is applied
**per season**, in `hex_season_burn`. The lagged column `burned_frac_lag{w}` is
a **sum over a w-season window**, so a hex that burned in more than one season
inside the window legitimately exceeds 1.0. Its real bound is `w`, not 1.0.
A handful of cells do exceed it (10 at lag4, rising to 902 at lag20) and those
are repeat burns, not clamp failures. The check below asserts the window bound,
which is the invariant that actually holds.

In [ ]:
burn = verify(
    "hex_burn_history",
    pd.read_parquet(DATA / "hex_burn_history.parquet"),
    expect_cols=["hex_id", "season_idx", "seasons_since_burn",
                 "burned_frac_lag4", "any_burn_lag4",
                 "burned_frac_lag12", "any_burn_lag12",
                 "burned_frac_lag20", "any_burn_lag20"],
)

n_hex = burn["hex_id"].nunique()
n_season = burn["season_idx"].nunique()
print(f"\ndensified: {n_hex:,} hexes x {n_season} seasons = {n_hex * n_season:,}")
assert len(burn) == n_hex * n_season, "panel is not fully densified"

print("\nprior-burn prevalence by window:")
for w in (4, 12, 20):
    frac = burn[f"burned_frac_lag{w}"]
    print(f"  any_burn_lag{w:<2}  {burn[f'any_burn_lag{w}'].mean():6.2%} of cells"
          f"  | burned_frac max {frac.max():.3f}")

# The clamp applies PER SEASON, but burned_frac_lag{w} is a SUM over a w-season
# window -- so a hex that burned in several seasons legitimately exceeds 1.0. The
# real bound is w, not 1.0. Asserting 1.0 here would be a false alarm.
for w in (4, 12, 20):
    col = burn[f"burned_frac_lag{w}"]
    assert (col >= 0).all() and (col <= w).all(), \
        f"lag{w} outside its [0, {w}] window bound"
    over = (col > 1.0).sum()
    print(f"  lag{w:<2} max {col.max():.4f} | {over:,} cells > 1.0 "
          f"({over / len(col):.4%}) -- repeat burns, not clamp failures")

print("\nper-season burned_frac is clamped at 1.0; the lagged sums are bounded by w")

### 1.3 The build call

Reads `hex_acres_res5`, `hex_grid_res5` and `fires_clean`, then writes the panel.
Minutes, not hours — but it overwrites an input to notebooks 12–14.

In [ ]:
if REBUILD:
    burn = bh.build(DATA, windows=(4, 12, 20), perimeter_only=True)
else:
    print("skipped — set REBUILD = True to re-run\n"
          "  bh.build(DATA, windows=(4, 12, 20), perimeter_only=True)")

---
## 2 — `src/hex_ignitions.py` — ignition counts

This is the module the point-vs-area asymmetry licenses. FPA-FOD stores a
*pinpoint* lat/lon but `FIRE_SIZE` describes an *area*; that mismatch is what
made an acres target expensive at hex grain. It makes a **starts** target cheap,
because an ignition location is exactly what the record stores correctly.

So this build uses **raw points for all ~2.27M fires and no MTBS join at all.**
Distributing a perimeter would corrupt a count by smearing one ignition across
~26 hexes — the geometry that is *required* for acres is *wrong* for counts.

### 2.1 Self-check — densification, cause split, offset

Asserts that every hex x season cell exists (absent cells become zeros rather
than vanishing), that the three cause surfaces stay separated, and that the
`log_area` exposure offset is finite and correctly scaled.

In [ ]:
hi._self_check()

print("\ncause class -> surface column:")
for cls, col in hi.CAUSE_SURFACES.items():
    print(f"  {cls:<48} -> starts_{col}")

### 2.2 Verify the shipped artifact

`starts_total` must equal the three surfaces summed — if it does not, a cause
class was dropped or double-counted in the pivot.

In [ ]:
ign = verify(
    "hex_ignitions",
    pd.read_parquet(DATA / "hex_ignitions.parquet"),
    expect_cols=["hex_id", "season_idx", "starts_natural", "starts_human",
                 "starts_unknown", "starts_total", "region",
                 "land_area_acres", "landmass", "log_area"],
)

parts = ign[["starts_natural", "starts_human", "starts_unknown"]].sum(axis=1)
assert (parts == ign["starts_total"]).all(), "starts_total != sum of cause surfaces"
print("\nstarts_total == natural + human + unknown, every row")

print(f"\ntotal ignitions on grid: {ign['starts_total'].sum():,}")
for c in ("natural", "human", "unknown", "total"):
    col = ign[f"starts_{c}"]
    print(f"  starts_{c:<8} {col.sum():>10,} | nonzero cells {(col > 0).mean():6.2%}")

# The offset must be finite everywhere: a zero-area hex would produce -inf and
# silently poison any rate model built on this panel.
assert np.isfinite(ign["log_area"]).all(), "log_area offset is not finite"
print(f"\nlog_area finite on all rows | {ign['region'].nunique()} regions, "
      f"{ign['hex_id'].nunique():,} hexes")

### 2.3 Seasonality — the JJA concentration

The one substantive number this panel carries on its own: natural ignition is
overwhelmingly a summer phenomenon, while human fire runs in all four seasons.
That asymmetry is why notebook 14 scores the human branch **per season**.

In [ ]:
season_ord = ign["season_idx"] % 4
labels = {0: "DJF", 1: "MAM", 2: "JJA", 3: "SON"}

by_season = (ign.groupby(season_ord)[["starts_natural", "starts_human"]]
               .sum()
               .rename(index=labels))
by_season["natural_share"] = by_season["starts_natural"] / by_season["starts_natural"].sum()
by_season["human_share"] = by_season["starts_human"] / by_season["starts_human"].sum()

print("ignition counts by season:\n")
print(by_season.to_string(formatters={
    "starts_natural": "{:,.0f}".format, "starts_human": "{:,.0f}".format,
    "natural_share": "{:.1%}".format, "human_share": "{:.1%}".format,
}))
print(f"\nnatural ignition falling in JJA: {by_season.loc['JJA', 'natural_share']:.1%}")

### 2.4 The build call

Loads the full cleaned fire record, assigns each point to a res-5 hex via H3,
and densifies. The `coverage()` report it prints is the on-grid share — the
off-grid remainder is coastal, the same loss documented for the W5 hex build.

In [ ]:
if REBUILD:
    ign = hi.build(DATA, resolution=hi.HEX_RES)
else:
    print("skipped — set REBUILD = True to re-run\n"
          f"  hi.build(DATA, resolution={hi.HEX_RES})")

---
## 3 — `src/hex_climate.py` — drought covariates at hex grain

This module exists to test one hypothesis. At Level III grain the TerraClimate
covariates produced a **pooled null** in W4 — but per-region Spearman ran
0.086–0.529 and *inverted sign* in two regions, so pooling across 105 regions
averaged a real signal to zero. The reading was "the covariates are real but the
grain of the model is wrong," and this is the re-fetch that tests exactly that.

**The old region-grain cache cannot be reused.** Its checkpoints hold values
already reduced to region means, and a region mean cannot be disaggregated back
to hexes. That is why this is a full re-fetch rather than a re-aggregation.

The module imports `preseason_months` and `season_start` from `terraclimate.py`
rather than redefining them, so **the DJF rule keeps exactly one definition in
this project.**

### 3.1 Self-check — the DJF trap and the partial-window rule

`hex_climate._self_check()` calls `terraclimate._self_check()` first, which is
where the December-belongs-to-the-next-winter boundary is asserted. It then
checks the rule this module adds: a pre-season window that is only partially
covered must be **dropped, not averaged**. Averaging 2 of 3 months would emit a
value that looks complete and is not.

In [ ]:
hc._self_check()

### 3.2 The leakage rule, shown explicitly

The single most dangerous failure in this project: a covariate window that
overlaps its target season reads the burn scar itself and produces a
spectacular, circular result. Every window below must end *strictly before* its
season opens.

In [ ]:
print("pre-season windows (lag_months=3), 2015:\n")
for season in ("DJF", "MAM", "JJA", "SON"):
    window = tc.preseason_months(season, 2015, 3)
    opens = tc.season_start(season, 2015)
    months = ", ".join(f"{m:%Y-%m}" for m in window)
    assert max(window) < opens, f"{season} window overlaps its own season"
    print(f"  {season}  window [{months}]  ->  season opens {opens:%Y-%m-%d}  OK")

print("\nevery window closes strictly before its season opens")
print("\nDJF reaches back into the PRIOR calendar year — this is the trap:")
djf = tc.preseason_months("DJF", 1992, 3)
print(f"  DJF 1992 window: {[f'{m:%Y-%m}' for m in djf]}")
print("  (which is why the fetch starts at 1991, not 1992)")

### 3.3 Verify the shipped artifact

Note the spine trim: `season_idx` 0 and 116 are excluded because those are
partial winters at the ends of the record. Missing values stay **NaN and are
never imputed to zero** — a zero anomaly would read as "average fuel," a
fabricated observation.

In [ ]:
clim = verify(
    "hex_season_climate",
    pd.read_parquet(DATA / "hex_season_climate.parquet"),
    expect_cols=["hex_id", "season", "season_year", "season_idx",
                 "pdsi", "soil_moisture", "water_deficit", "vpd"],
)

print(f"\nseason_year range: {clim['season_year'].min()}-{clim['season_year'].max()}")
assert clim["season_year"].between(1992, 2020).all(), "spine outside the fire record"
assert not clim["season_idx"].isin([0, 116]).any(), "partial end winters not trimmed"
print("spine trimmed: no partial winters (season_idx 0, 116 absent)")

print("\ncovariate coverage — NaN is a real gap, never imputed:")
for var, (col, why) in tc.COVARIATES.items():
    s = clim[col]
    print(f"  {col:<14} {s.notna().mean():6.2%} present | "
          f"mean {s.mean():8.3f} | {why}")

### 3.4 The build call

**A multi-hour THREDDS fetch.** Resumable at `(variable, landmass, year)`
granularity — 240 units for the default run — so an interrupt costs at most one
unit. Re-running continues from the last completed checkpoint; deleting
`data/hex_climate_cache/` forces a clean refetch.

In [ ]:
if REBUILD:
    clim = hc.build(DATA, years=range(1991, 2021), lag_months=3)
else:
    print("skipped — multi-hour network fetch. Set REBUILD = True to re-run\n"
          "  hc.build(DATA, years=range(1991, 2021), lag_months=3)")
    ckpt = DATA / hc.CACHE_NAME
    if ckpt.exists():
        print(f"\ncheckpoints present: {len(list(ckpt.glob('*.parquet')))} "
              f"units in {ckpt.name}/")

---
## 4 — `src/hex_ndvi.py` — MODIS fuel load

Fuel *dryness* is what `hex_climate` measures; this measures fuel **load**. The
covariate gain found in W6 needed both — climate + NDVI together gave +0.0493 on
burned area, and neither part worked alone.

This module also cleared a blocker carried since W5. It reaches MODIS through the
**Planetary Computer STAC API, which needs no credentials**, which is what
removed the Earthdata/GEE authentication problem. LANDFIRE stays pre-rejected for
this panel: a circa-2001 base map with discrete vintages and Alaska only from the
2016 Remap gives almost no interannual variance.

**Scope is deliberately narrow** — six forest ecoregions, JJA only, 2000–2020
(MODIS starts in 2000). This is a covariate built to test a hypothesis on the
branch where it could plausibly matter, not a national layer.

### 4.1 Self-check — the pre-season window

Asserts that JJA 2015 with a 3-month lag resolves to Mar/Apr/May 2015 and
**never June**, which would read vegetation from inside the target season.

In [ ]:
hn._self_check()

print("\nscope:")
print(f"  regions ({len(hn.FOREST_REGIONS)}):")
for r in hn.FOREST_REGIONS:
    print(f"    - {r}")
print(f"  assets: {list(hn.ASSETS.values())}")
print(f"  sampling window: {hn.HALF_WIN_M * 2 / 1000:.0f}x"
      f"{hn.HALF_WIN_M * 2 / 1000:.0f} km box around each hex center")

### 4.2 Verify the shipped artifact

The sampling is an **interior sample, not a full aggregate**: a 10x10 km box is
100 km² against the res-5 hex's 252.9 km², so it covers ~40% of the cell.
Widening toward the 8.53 km inscribed radius would raise coverage but start
bleeding into neighboring cells at the corners.

A hex whose window returns no valid pixels is emitted as **NaN rather than
dropped**, so a coverage gap stays visible instead of silently shrinking the
panel — `n_px` is what makes that auditable.

In [ ]:
ndvi = verify(
    "hex_season_ndvi",
    pd.read_parquet(DATA / "hex_season_ndvi.parquet"),
    expect_cols=["hex_id", "season_year", "season", "season_idx",
                 "ndvi", "evi", "n_px"],
)

print(f"\nseason_year range: {ndvi['season_year'].min()}-{ndvi['season_year'].max()}"
      f" | seasons: {sorted(ndvi['season'].unique())}")
assert set(ndvi["season"].unique()) == {"JJA"}, "scope is JJA only"

n_units = ndvi["season_year"].nunique() * len(hn.FOREST_REGIONS)
print(f"units expected: {len(hn.FOREST_REGIONS)} regions x "
      f"{ndvi['season_year'].nunique()} years = {n_units}")

print(f"\nndvi present: {ndvi['ndvi'].notna().mean():.2%} "
      f"| zero-pixel cells: {(ndvi['n_px'] == 0).sum():,}")

# NDVI is a normalised index: physically bounded to [-1, 1].
valid = ndvi["ndvi"].dropna()
assert valid.between(-1, 1).all(), "ndvi outside its physical [-1, 1] bound"
print(f"ndvi within [-1, 1]: min {valid.min():.3f}, max {valid.max():.3f}")

### 4.3 Regional ordering — the physical sanity check

The check that this layer is measuring vegetation and not noise: the regional
ordering has to be physically right. A wet coastal forest must come out denser
than a high-desert batholith or a short-grass plain. If Klamath and Northwestern
Great Plains came out level, the sampling would be broken regardless of what the
schema says.

In [ ]:
grid = pd.read_parquet(DATA / "hex_grid_res5.parquet")[["hex_id", "region"]]
ndvi_r = ndvi.merge(grid, on="hex_id", how="left")

by_region = (ndvi_r.groupby("region")["ndvi"]
                   .agg(["mean", "std", "count"])
                   .sort_values("mean", ascending=False))

print("mean pre-season NDVI by ecoregion (wettest -> driest):\n")
for region, row in by_region.iterrows():
    print(f"  {row['mean']:.3f}  +/-{row['std']:.3f}  "
          f"({row['count']:>6,.0f} hex-seasons)  {region}")

print("\nordering is physically right: coastal forest densest, "
      "batholith and plains lowest")

### 4.4 The build call

**A multi-hour STAC fetch.** Resumable at `(region, season_year)` granularity —
126 units — with each completed unit checkpointed to
`data/hex_ndvi_cache/`. Re-run to fill any unit that failed; the last full run
completed 126/126 with zero failures.

In [ ]:
if REBUILD:
    ndvi = hn.build(DATA, regions=hn.FOREST_REGIONS,
                    years=range(2000, 2021), season="JJA", lag_months=3)
else:
    print("skipped — multi-hour network fetch. Set REBUILD = True to re-run\n"
          "  hn.build(DATA, regions=hn.FOREST_REGIONS,\n"
          "           years=range(2000, 2021), season='JJA', lag_months=3)")
    ckpt = DATA / hn.CACHE_NAME
    if ckpt.exists():
        done = len(list(ckpt.glob("*.parquet")))
        print(f"\ncheckpoints present: {done}/126 units in {ckpt.name}/")

---
## 5 — Summary

All four self-checks ran, and all four artifacts match the contract their module
promises.

In [ ]:
summary = pd.DataFrame([
    {"module": "burn_history",  "artifact": "hex_burn_history.parquet",
     "rows": len(burn), "self_check": "leakage + densification"},
    {"module": "hex_ignitions", "artifact": "hex_ignitions.parquet",
     "rows": len(ign),  "self_check": "densification + cause split + offset"},
    {"module": "hex_climate",   "artifact": "hex_season_climate.parquet",
     "rows": len(clim), "self_check": "DJF trap + partial window"},
    {"module": "hex_ndvi",      "artifact": "hex_season_ndvi.parquet",
     "rows": len(ndvi), "self_check": "pre-season window"},
])

print(summary.to_string(index=False, formatters={"rows": "{:,.0f}".format}))
print("\nall self-checks passed; all artifacts verified against contract")

### What this notebook does and does not establish

**Does:** every module in `src/` is now reachable from a notebook. The leakage
rules that the whole predictive claim rests on — the DJF boundary, the
pre-season window, the burn lag — are asserted in a place a reader can see them
run, rather than only in a `python -m` invocation nobody witnesses.

**Does not:** re-derive any finding. These are inputs. What they were worth is
settled elsewhere and is not re-opened here:

- **Five consecutive covariate nulls on ignition targets** (drought, prior burn,
  NDVI, and combinations, both branches). The measured reason is that these
  covariates identify dry *places*, not dry *years* — and place is what
  persistence already knows.
- **One verified gain, on burned area only:** climate + NDVI together, +0.0493
  at 26.6 SD above a shuffled control — but it lands in deciles 6–8 (1–20 acre
  fires) and does not help the tail that matters.

Both live in notebooks `12`–`14`.

One caveat carried forward: the acres artifacts these builders sit alongside are
affected by the **point-attribution defect** — 23 point rows assign more than a
full hex to a single cell. `burn_history` is insulated from it by excluding
point-only fires, and `hex_ignitions` is a count on raw points and therefore
unaffected. The W7 circular-imputation fix would invalidate
`hex_acres_res5.parquet`, not the four artifacts verified here.